In [12]:
import cdd

# H-representation: b + a1*x1 + ... >= 0
# Inequality: 5/2 v0 <= 1  -->  1 - 5/2 v0 >= 0

ineq = [
    [1, -5/2, 0, 0, 0],  # main inequality
]

# bounds: 0 <= v_i <= 1  -->  two inequalities per variable
for i in range(4):
    # v_i >= 0  --> 0 + 1*v_i >= 0
    row = [0] + [0]*4
    row[i+1] = 1
    ineq.append(row)
    # v_i <= 1  --> 1 - 1*v_i >= 0
    row = [1] + [0]*4
    row[i+1] = -1
    ineq.append(row)

print(ineq)
# Create cdd.Matrix
mat = cdd.matrix_from_array(ineq)
mat.rep_type = cdd.RepType.INEQUALITY

# # Create polyhedron
poly = cdd.polyhedron_from_matrix(mat)
# Compute generators (vertices)
generators = cdd.copy_generators(poly).array
vertices = [g[1:] for g in generators]
for g in vertices:
    print(g)

filtered = []
for v in vertices:
    if generator_filter(v, [5/2,0,0,0], 1):   # inequality tight
        filtered.append(v)

print(filtered)

[[1, -2.5, 0, 0, 0], [0, 1, 0, 0, 0], [1, -1, 0, 0, 0], [0, 0, 1, 0, 0], [1, 0, -1, 0, 0], [0, 0, 0, 1, 0], [1, 0, 0, -1, 0], [0, 0, 0, 0, 1], [1, 0, 0, 0, -1]]
[0.4, 1.0, 1.0, 0.0]
[0.0, 1.0, 1.0, 0.0]
[0.4, 0.0, 1.0, 0.0]
[0.0, 0.0, 1.0, 0.0]
[0.0, 0.0, 0.0, 0.0]
[0.4, 0.0, 0.0, 0.0]
[0.0, 1.0, 0.0, 0.0]
[0.4, 1.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 1.0]
[0.4, 0.0, 0.0, 1.0]
[0.0, 1.0, 0.0, 1.0]
[0.4, 1.0, 0.0, 1.0]
[0.0, 0.0, 1.0, 1.0]
[0.4, 0.0, 1.0, 1.0]
[0.0, 1.0, 1.0, 1.0]
[0.4, 1.0, 1.0, 1.0]
[[0.4, 1.0, 1.0, 0.0], [0.4, 0.0, 1.0, 0.0], [0.4, 0.0, 0.0, 0.0], [0.4, 1.0, 0.0, 0.0], [0.4, 0.0, 0.0, 1.0], [0.4, 1.0, 0.0, 1.0], [0.4, 0.0, 1.0, 1.0], [0.4, 1.0, 1.0, 1.0]]


In [11]:
from itertools import product

def compute_generators(r, r0):
    n = len(r)
    T = [i for i in range(n) if r[i] > 0] # Only non-zeroes
    Z = [i for i in range(n) if r[i] <= 0] # Only zeroes

    generators = []

    for s_star in T:

        others = [s for s in T if s != s_star]

        for assign in product([0,1], repeat=len(others)):
            d = [0]*n
            sum_fixed = 0

            for s,val in zip(others,assign):
                d[s] = val
                sum_fixed += r[s]*val

            d_star = (r0 - sum_fixed)/r[s_star]

            if 0 <= d_star <= 1:
                d[s_star] = d_star

                generators.append(d.copy())
    print(generators)
    # expand zero-coefficient coordinates
    final = []
    for g in generators:
        for assign in product([0,1], repeat=len(Z)):
            v = g.copy()
            for s,val in zip(Z,assign):
                v[s] = val
            final.append(v)

    return final

compute_generators([5/2,0,0,0], 1)

[[0.4, 0, 0, 0]]


[[0.4, 0, 0, 0],
 [0.4, 0, 0, 1],
 [0.4, 0, 1, 0],
 [0.4, 0, 1, 1],
 [0.4, 1, 0, 0],
 [0.4, 1, 0, 1],
 [0.4, 1, 1, 0],
 [0.4, 1, 1, 1]]

In [12]:
def is_binary(x, eps=1e-9):
    return abs(x) < eps or abs(x-1) < eps

def generator_filter(v, r, r0, eps=1e-9):

    # 1. inequality must be tight
    if abs(sum(r[i]*v[i] for i in range(len(v))) - r0) > eps:
        return False

    # 2. count fractional coordinates
    fractional = sum(not is_binary(x) for x in v)

    return fractional <= 1

In [13]:
compute_generators([1,1/7], 1)

[[1.0, 0], [0.8571428571428572, 1], [1, 0.0]]


[[1.0, 0], [0.8571428571428572, 1], [1, 0.0]]